# Experiment 2: Linear Mixed Models

## Theoretical Framework

**Core effects** (can exist as main effects):
- **CT**: Central Tendency (curDur) - direct effect on bias
- **SD**: Serial Dependence (preDur) - direct effect on bias

**Modulators** (only exist as interactions):
- **curCoh**: Current uncertainty - modulates SD strength (weighting)
- **preResp**: Previous response - modulated BY gating (decision carryover)
- **SameSwitch**: Gating factor - modulates integration of prior info

## Experiment 2 Hypothesis (Gating)

**Question**: Does categorical discontinuity (SameSwitch) gate decision carryover (preResp)?

**Key interaction**: `preResp × SameSwitch`

## Variable Coding
- Duration: centered at 1.2s
- Coherence: original values (0.3, 0.7)
- preResp: Long=1, Short=0  
- SameSwitch: Same=0, Switch=1

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'

## 1. Data Preparation

In [2]:
def prepare_lmm_data(df):
    """
    Prepare variables for LMM analysis.
    
    Coding:
    - Duration: centered at 1.2s
    - Coherence: original values (0.3, 0.7) - NOT centered
    - preResp: Long=1, Short=0
    - SameSwitch: Same=0, Switch=1
    """
    df = df.copy()
    
    # Remove outliers
    df = df[df['is_outlier'] == False].copy()
    
    # Center duration variables
    df['curDur'] = df['curDur'] - 1.2
    df['preDur'] = df['preDur1back'] - 1.2
    
    # Coherence: original values (NOT centered)
    df['curCoh'] = df['curCoherence']
    df['preCoh'] = df['preCoherence1back']
    
    # SameSwitch: Same=0, Switch=1
    df['SameSwitch'] = (df['preCoherence1back'] != df['curCoherence']).astype(int)
    
    # preResp: previous response (Long=1, Short=0)
    if 'preResp' not in df.columns:
        df['preResp'] = df.groupby('subID')['resp_type'].shift(1)
    df['preResp'] = (df['preResp'] == 'Long').astype(float)
    
    # Drop missing values
    df = df.dropna(subset=['preDur', 'preCoherence1back', 'preResp'])
    
    # Summary
    print(f"Data prepared: {len(df)} trials, {df['subID'].nunique()} subjects")
    print(f"\nVariables:")
    print(f"  curDur: [{df['curDur'].min():.2f}, {df['curDur'].max():.2f}] (centered)")
    print(f"  preDur: [{df['preDur'].min():.2f}, {df['preDur'].max():.2f}] (centered)")
    print(f"  curCoh: {sorted(df['curCoh'].unique())}")
    print(f"  SameSwitch: {df['SameSwitch'].value_counts().to_dict()}")
    print(f"  preResp: {df['preResp'].value_counts().to_dict()}")
    
    return df

In [3]:
# Load and prepare data
df_raw = pd.read_pickle('../../data/experiment2/E2.pkl')
df_lmm = prepare_lmm_data(df_raw)

Data prepared: 4659 trials, 22 subjects

Variables:
  curDur: [-0.40, 0.40] (centered)
  preDur: [-0.40, 0.40] (centered)
  curCoh: [np.float64(0.3), np.float64(0.7)]
  SameSwitch: {1: 2492, 0: 2167}
  preResp: {0.0: 2522, 1.0: 2137}


## 2. Model Definitions

### Theoretical Structure

| Variable | Role | Rationale |
|----------|------|-----------|
| curDur | Main effect | CT directly biases reproduction |
| preDur | Main effect | SD directly biases reproduction |
| curCoh | Modulator only | Uncertainty doesn't bias directly; it modulates SD |
| preResp | Modulator only | Decision carryover is gated by SameSwitch |
| SameSwitch | Interaction only | Gating modulates prior integration |

### Model Hierarchy

```
Level 0: Null
Level 1: Core (CT + SD)
Level 2: + Gating interaction (preResp × SameSwitch) ← Manuscript LMM-E1
Level 3: + Weighting interaction (preDur × curCoh)
Level 4: Full model
```

In [4]:
def get_model_formulas():
    """
    Experiment 2 model hierarchy.
    
    Theoretical constraints:
    - curCoh, preResp: modulators only (no main effects)
    - SameSwitch: gating factor (interaction only)
    """
    formulas = {
        # ===== Level 0: Null =====
        'M0_Null': 'curBias ~ 1',
        
        # ===== Level 1: Core Effects =====
        'M1_CT': 'curBias ~ curDur',
        'M2_SD': 'curBias ~ preDur',
        'M3_Core': 'curBias ~ curDur + preDur',
        
        # ===== Level 2: Gating Hypothesis =====
        # preResp × SameSwitch: decision carryover modulated by gating
        'M4_Gating': 'curBias ~ curDur + preDur + preResp:SameSwitch',
        
        # ===== Level 3: Weighting Hypothesis =====
        # preDur × curCoh: SD modulated by uncertainty
        'M5_Weighting': 'curBias ~ curDur + preDur + preDur:curCoh',
        
        # ===== Level 4: Both Interactions =====
        'M6_Both': 'curBias ~ curDur + preDur + preResp:SameSwitch + preDur:curCoh',
        
        # ===== Level 5: SD × Gating =====
        # preDur × SameSwitch: SD itself modulated by gating
        'M7_SD_Gating': 'curBias ~ curDur + preDur + preDur:SameSwitch',
        
        # ===== Level 6: Full Model (with main effects) =====
        'M8_Full': 'curBias ~ curDur + preDur + preResp + SameSwitch + preResp:SameSwitch + curCoh',
    }
    return formulas

## 3. Model Fitting Functions

In [5]:
def fit_lmm(df, formulas=None, method='lbfgs', reml=True, verbose=True):
    """
    Fit all hierarchical models using REML.
    """
    if formulas is None:
        formulas = get_model_formulas()
    
    results = {}
    for name, formula in formulas.items():
        try:
            model = smf.mixedlm(formula, df, groups=df['subID'])
            result = model.fit(method=method, reml=reml, disp=False)
            
            # Check for convergence issues
            if np.isinf(result.llf) or np.isnan(result.llf):
                if verbose:
                    print(f"  [WARN] {name}: LogLik is inf/nan, trying alternative methods...")
                for alt_method in ['powell', 'cg', 'nm']:
                    try:
                        result = model.fit(method=alt_method, reml=reml, disp=False)
                        if not np.isinf(result.llf) and not np.isnan(result.llf):
                            if verbose:
                                print(f"       -> Succeeded with {alt_method}")
                            break
                    except:
                        continue
            
            results[name] = result
            if verbose:
                status = "OK" if not np.isinf(result.llf) else "WARN (inf)"
                print(f"  [{status}] {name}")
        except Exception as e:
            if verbose:
                print(f"  [FAIL] {name}: {e}")
            results[name] = None
    
    return results


def compute_model_fit(result, n_obs):
    """
    Compute AIC and BIC manually.
    """
    if result is None:
        return np.nan, np.nan
    
    k = len(result.params)
    llf = result.llf
    
    if np.isinf(llf) or np.isnan(llf):
        return np.nan, np.nan
    
    aic = 2 * k - 2 * llf
    bic = k * np.log(n_obs) - 2 * llf
    
    return aic, bic


def compare_models(results, n_obs):
    """Create model comparison table."""
    comparison = []
    
    for name, res in results.items():
        if res is None:
            comparison.append({
                'Model': name,
                'nParams': np.nan,
                'LogLik': np.nan,
                'AIC': np.nan,
                'BIC': np.nan,
            })
            continue
        
        aic, bic = compute_model_fit(res, n_obs)
        
        comparison.append({
            'Model': name,
            'nParams': len(res.params),
            'LogLik': res.llf,
            'AIC': aic,
            'BIC': bic,
        })
    
    comp_df = pd.DataFrame(comparison)
    
    # Compute delta AIC/BIC
    valid_aic = comp_df['AIC'].dropna()
    valid_bic = comp_df['BIC'].dropna()
    
    if len(valid_aic) > 0:
        comp_df['dAIC'] = comp_df['AIC'] - valid_aic.min()
    else:
        comp_df['dAIC'] = np.nan
        
    if len(valid_bic) > 0:
        comp_df['dBIC'] = comp_df['BIC'] - valid_bic.min()
    else:
        comp_df['dBIC'] = np.nan
    
    return comp_df

## 4. Fit All Models

In [6]:
print("Fitting hierarchical models (REML estimation)...\n")
lmm_results = fit_lmm(df_lmm, reml=True)

Fitting hierarchical models (REML estimation)...

  [OK] M0_Null
  [OK] M1_CT
  [OK] M2_SD
  [OK] M3_Core
  [OK] M4_Gating
  [OK] M5_Weighting
  [OK] M6_Both
  [OK] M7_SD_Gating
  [OK] M8_Full


## 5. Model Comparison

In [7]:
def print_comparison_table(comp_df, title="MODEL COMPARISON"):
    """Print formatted comparison table."""
    print("=" * 100)
    print(title)
    print("=" * 100)
    print(f"{'Model':<20} {'k':>4} {'LogLik':>12} {'AIC':>12} {'dAIC':>8} {'BIC':>12} {'dBIC':>8}")
    print("-" * 100)
    
    for _, row in comp_df.iterrows():
        if pd.isna(row['LogLik']):
            print(f"{row['Model']:<20} {'--':>4} {'--':>12} {'--':>12} {'--':>8} {'--':>12} {'--':>8}")
        else:
            print(f"{row['Model']:<20} {int(row['nParams']):>4} {row['LogLik']:>12.2f} {row['AIC']:>12.2f} {row['dAIC']:>8.2f} {row['BIC']:>12.2f} {row['dBIC']:>8.2f}")
    
    print("=" * 100)
    
    # Find best models
    valid = comp_df.dropna(subset=['AIC'])
    if len(valid) > 0:
        best_aic = valid.loc[valid['AIC'].idxmin(), 'Model']
        best_bic = valid.loc[valid['BIC'].idxmin(), 'Model']
        print(f"\nBest by AIC: {best_aic}")
        print(f"Best by BIC: {best_bic}")

In [8]:
# Create comparison table
n_obs = len(df_lmm)
comp_df = compare_models(lmm_results, n_obs)
print_comparison_table(comp_df, title="MODEL COMPARISON - EXPERIMENT 2")

MODEL COMPARISON - EXPERIMENT 2
Model                   k       LogLik          AIC     dAIC          BIC     dBIC
----------------------------------------------------------------------------------------------------
M0_Null                 2      -219.81       443.62  1642.05       456.52  1603.37
M1_CT                   3       542.80     -1079.60   118.83     -1060.26    86.60
M2_SD                   3      -197.10       400.21  1598.63       419.55  1566.40
M3_Core                 4       572.94     -1137.87    60.55     -1112.09    34.77
M4_Gating               5       576.64     -1143.29    55.14     -1111.06    35.80
M5_Weighting            5       571.58     -1133.15    65.27     -1100.92    45.93
M6_Both                 6       575.25     -1138.50    59.92     -1099.83    47.03
M7_SD_Gating            5       570.79     -1131.57    66.85     -1099.34    47.51
M8_Full                 8       607.21     -1198.43     0.00     -1146.85     0.00

Best by AIC: M8_Full
Best by BIC: M8

In [9]:
# Sort by AIC and display
print("\nModels Ranked by AIC:")
comp_sorted = comp_df.dropna(subset=['AIC']).sort_values('AIC')
display(comp_sorted[['Model', 'nParams', 'LogLik', 'AIC', 'dAIC', 'BIC', 'dBIC']].round(2))


Models Ranked by AIC:


,Model,nParams,LogLik,AIC,dAIC,BIC,dBIC
8,M8_Full,8,607.21,-1198.43,0.00,-1146.85,0.00
4,M4_Gating,5,576.64,-1143.29,55.14,-1111.06,35.80
6,M6_Both,6,575.25,-1138.50,59.92,-1099.83,47.03
3,M3_Core,4,572.94,-1137.87,60.55,-1112.09,34.77
5,M5_Weighting,5,571.58,-1133.15,65.27,-1100.92,45.93
7,M7_SD_Gating,5,570.79,-1131.57,66.85,-1099.34,47.51
1,M1_CT,3,542.80,-1079.60,118.83,-1060.26,86.60
2,M2_SD,3,-197.10,400.21,1598.63,419.55,1566.40
0,M0_Null,2,-219.81,443.62,1642.05,456.52,1603.37


## 6. Best Model Details

In [10]:
def print_model_summary(result, model_name="Model"):
    """Print detailed model results."""
    if result is None:
        print(f"{model_name}: Model failed to fit")
        return
    
    print(f"\n{'=' * 90}")
    print(f"{model_name}")
    print(f"{'=' * 90}")
    print(f"{'Parameter':<40} {'Beta':>10} {'SE':>10} {'z':>10} {'p':>12}")
    print("-" * 90)
    
    for param in result.params.index:
        if param == 'Group Var':
            continue
        beta = result.params[param]
        se = result.bse.get(param, np.nan)
        z = result.tvalues.get(param, np.nan)
        p = result.pvalues.get(param, np.nan)
        
        if pd.isna(p):
            sig, p_str = '', 'NA'
        elif p < 0.001:
            sig, p_str = '***', '< .001'
        elif p < 0.01:
            sig, p_str = '**', f'{p:.4f}'
        elif p < 0.05:
            sig, p_str = '*', f'{p:.4f}'
        else:
            sig, p_str = '', f'{p:.4f}'
        
        print(f"{param:<40} {beta:>10.4f} {se:>10.4f} {z:>10.2f} {p_str:>10} {sig}")
    
    print("-" * 90)
    print(f"Random Intercept Var: {result.cov_re.iloc[0,0]:.4f}")
    print(f"N obs: {int(result.nobs)}, N groups: {len(result.random_effects)}")
    aic, bic = compute_model_fit(result, result.nobs)
    print(f"AIC: {aic:.2f}, BIC: {bic:.2f}")

In [11]:
# Print the theoretically-driven gating model
print_model_summary(lmm_results['M4_Gating'], "M4_Gating (Theory): curDur + preDur + preResp:SameSwitch")


M4_Gating (Theory): curDur + preDur + preResp:SameSwitch
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                   -0.0260     0.0149      -1.75     0.0799 
curDur                                      -0.4780     0.0111     -42.90     < .001 ***
preDur                                       0.0814     0.0113       7.24     < .001 ***
preResp:SameSwitch                           0.0290     0.0074       3.93     < .001 ***
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0046
N obs: 4659, N groups: 22
AIC: -1143.29, BIC: -1111.06


In [12]:
# Print full model (with all main effects)
print_model_summary(lmm_results['M8_Full'], "M8_Full: curDur + preDur + preResp + SameSwitch + preResp:SameSwitch + curCoh")


M8_Full: curDur + preDur + preResp + SameSwitch + preResp:SameSwitch + curCoh
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                   -0.0730     0.0176      -4.14     < .001 ***
curDur                                      -0.4784     0.0110     -43.31     < .001 ***
preDur                                       0.0530     0.0116       4.56     < .001 ***
preResp                                      0.0808     0.0094       8.60     < .001 ***
SameSwitch                                   0.0174     0.0084       2.07     0.0382 *
preResp:SameSwitch                          -0.0354     0.0124      -2.86     0.0043 **
curCoh                                       0.0334     0.0154       2.17     0.0298 *
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0047
N obs

In [13]:
# Print the actual best model by AIC
valid = comp_df.dropna(subset=['AIC'])
if len(valid) > 0:
    best_aic_name = valid.loc[valid['AIC'].idxmin(), 'Model']
    best_bic_name = valid.loc[valid['BIC'].idxmin(), 'Model']
    
    print_model_summary(lmm_results[best_aic_name], f"{best_aic_name} (Best by AIC)")
    
    if best_bic_name != best_aic_name:
        print_model_summary(lmm_results[best_bic_name], f"{best_bic_name} (Best by BIC)")


M8_Full (Best by AIC)
Parameter                                      Beta         SE          z            p
------------------------------------------------------------------------------------------
Intercept                                   -0.0730     0.0176      -4.14     < .001 ***
curDur                                      -0.4784     0.0110     -43.31     < .001 ***
preDur                                       0.0530     0.0116       4.56     < .001 ***
preResp                                      0.0808     0.0094       8.60     < .001 ***
SameSwitch                                   0.0174     0.0084       2.07     0.0382 *
preResp:SameSwitch                          -0.0354     0.0124      -2.86     0.0043 **
curCoh                                       0.0334     0.0154       2.17     0.0298 *
------------------------------------------------------------------------------------------
Random Intercept Var: 0.0047
N obs: 4659, N groups: 22
AIC: -1198.43, BIC: -1146.85


## 7. Summary for Manuscript

In [14]:
# Model comparison
print("\n" + "=" * 80)
print("MODEL COMPARISON - EXPERIMENT 2")
print("=" * 80)

key_models = ['M3_Core', 'M4_Gating', 'M5_Weighting', 'M6_Both', 'M7_SD_Gating', 'M8_Full']
key_comp = comp_df[comp_df['Model'].isin(key_models)].sort_values('AIC')

print("\nKey Models (sorted by AIC):")
display(key_comp[['Model', 'nParams', 'LogLik', 'AIC', 'dAIC', 'BIC', 'dBIC']].round(2))


MODEL COMPARISON - EXPERIMENT 2

Key Models (sorted by AIC):


,Model,nParams,LogLik,AIC,dAIC,BIC,dBIC
8,M8_Full,8,607.21,-1198.43,0.00,-1146.85,0.00
4,M4_Gating,5,576.64,-1143.29,55.14,-1111.06,35.80
6,M6_Both,6,575.25,-1138.50,59.92,-1099.83,47.03
3,M3_Core,4,572.94,-1137.87,60.55,-1112.09,34.77
5,M5_Weighting,5,571.58,-1133.15,65.27,-1100.92,45.93
7,M7_SD_Gating,5,570.79,-1131.57,66.85,-1099.34,47.51


In [15]:
# Key findings
print("\n" + "=" * 80)
print("KEY FINDINGS - EXPERIMENT 2")
print("=" * 80)

# Gating interaction from M4_Gating
res_gating = lmm_results['M4_Gating']
print("\n1. Gating Model (M4_Gating):")
print(f"   preResp:SameSwitch interaction:")
for param in res_gating.params.index:
    if 'preResp' in param and 'SameSwitch' in param:
        b = res_gating.params[param]
        p = res_gating.pvalues[param]
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        print(f"   {param}: β = {b:.4f}, p = {p:.4f} {sig}")

# Full model
res_full = lmm_results['M8_Full']
print("\n2. Full Model (M8_Full):")
b = res_full.params.get('preResp:SameSwitch', np.nan)
p = res_full.pvalues.get('preResp:SameSwitch', np.nan)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f"   preResp:SameSwitch: β = {b:.4f}, p = {p:.4f} {sig}")

# Compare AIC
aic_gating, _ = compute_model_fit(res_gating, n_obs)
aic_full, _ = compute_model_fit(res_full, n_obs)
print(f"\n3. Model Fit Comparison:")
print(f"   M4_Gating (parsimonious): AIC = {aic_gating:.2f}")
print(f"   M8_Full (with main effects): AIC = {aic_full:.2f}")


KEY FINDINGS - EXPERIMENT 2

1. Gating Model (M4_Gating):
   preResp:SameSwitch interaction:
   preResp:SameSwitch: β = 0.0290, p = 0.0001 ***

2. Full Model (M8_Full):
   preResp:SameSwitch: β = -0.0354, p = 0.0043 **

3. Model Fit Comparison:
   M4_Gating (parsimonious): AIC = -1143.29
   M8_Full (with main effects): AIC = -1198.43


In [16]:
# Save results
comp_df.to_csv('lmm_model_comparison.csv', index=False)
print("Model comparison saved to lmm_model_comparison.csv")

Model comparison saved to lmm_model_comparison.csv
